# 03 暴露與疾病的關聯 — 參考解答

松柏護理之家退伍軍人症群聚事件練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, fisher_exact
from epi_learning.metrics import risk_ratio, odds_ratio

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

## 題目 1：COPD × 感染的完整分析

In [ ]:
ct_copd = pd.crosstab(
    df["comorbidity_copd"], df["infected"],
    margins=True, margins_name="合計",
)
ct_copd.index = ["無 COPD", "有 COPD", "合計"]
ct_copd.columns = ["未感染", "感染", "合計"]
print(ct_copd)

a = int(ct_copd.loc["有 COPD", "感染"])
b = int(ct_copd.loc["有 COPD", "未感染"])
c = int(ct_copd.loc["無 COPD", "感染"])
d = int(ct_copd.loc["無 COPD", "未感染"])

rr = risk_ratio(a, a + b, c, c + d)
or_val = odds_ratio(a, b, c, d)

ln_rr = np.log(rr)
se_rr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
ci_rr_lo = np.exp(ln_rr - 1.96 * se_rr)
ci_rr_hi = np.exp(ln_rr + 1.96 * se_rr)

ln_or = np.log(or_val)
se_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_or_lo = np.exp(ln_or - 1.96 * se_or)
ci_or_hi = np.exp(ln_or + 1.96 * se_or)

chi2, p, _, _ = chi2_contingency([[a, b], [c, d]])

print(f"\nCOPD → 感染")
print(f"  RR = {rr:.3f} (95% CI: {ci_rr_lo:.3f} – {ci_rr_hi:.3f})")
print(f"  OR = {or_val:.3f} (95% CI: {ci_or_lo:.3f} – {ci_or_hi:.3f})")
print(f"  卡方 = {chi2:.3f}, p-value = {p:.4f}")
print(f"  RR vs OR 差距: {abs(or_val - rr):.3f}（侵襲率高時 OR > RR）")

if ci_rr_lo > 1:
    print("  → COPD 是感染的統計顯著危險因子")
else:
    print("  → COPD 與感染無統計顯著關聯（CI 包含 1）")

## 題目 2：各共病的 RR / OR 排名

In [ ]:
comorbidities = [
    "comorbidity_chf", "comorbidity_dm",
    "comorbidity_cancer", "comorbidity_copd",
    "immunosuppressed",
]

results = []
for factor in comorbidities:
    ct = pd.crosstab(df[factor], df["infected"])
    a_i = int(ct.loc[1, 1])
    b_i = int(ct.loc[1, 0])
    c_i = int(ct.loc[0, 1])
    d_i = int(ct.loc[0, 0])
    rr_i = risk_ratio(a_i, a_i + b_i, c_i, c_i + d_i)
    or_i = odds_ratio(a_i, b_i, c_i, d_i)
    chi2_i, p_i, _, _ = chi2_contingency([[a_i, b_i], [c_i, d_i]])
    ln_rr_i = np.log(rr_i)
    se_i = np.sqrt(1/a_i - 1/(a_i+b_i) + 1/c_i - 1/(c_i+d_i))
    ci_lo = np.exp(ln_rr_i - 1.96 * se_i)
    ci_hi = np.exp(ln_rr_i + 1.96 * se_i)
    results.append({
        "共病": factor.replace("comorbidity_", "").upper()
                if "comorbidity_" in factor else factor.upper(),
        "RR": round(rr_i, 3),
        "95% CI": f"{ci_lo:.3f}–{ci_hi:.3f}",
        "OR": round(or_i, 3),
        "p-value": round(p_i, 4),
        "顯著": "*" if ci_lo > 1 else "",
        "RR-OR差": round(abs(or_i - rr_i), 3),
    })

rr_df = pd.DataFrame(results).sort_values("RR", ascending=False)
print("=== 各共病 RR / OR 排名 ===")
print(rr_df.to_string(index=False))
print("\n→ RR 最高的共病侵襲率也最高，因此 OR 偏離 RR 最多")

## 題目 3：性別差異分析

In [ ]:
df["is_male"] = (df["sex"] == "M").astype(int)

ct_sex = pd.crosstab(df["is_male"], df["infected"])
a_s = int(ct_sex.loc[1, 1])
b_s = int(ct_sex.loc[1, 0])
c_s = int(ct_sex.loc[0, 1])
d_s = int(ct_sex.loc[0, 0])

rr_sex = risk_ratio(a_s, a_s + b_s, c_s, c_s + d_s)
or_sex = odds_ratio(a_s, b_s, c_s, d_s)
chi2_s, p_s, _, _ = chi2_contingency([[a_s, b_s], [c_s, d_s]])

ln_rr_s = np.log(rr_sex)
se_s = np.sqrt(1/a_s - 1/(a_s+b_s) + 1/c_s - 1/(c_s+d_s))
ci_lo_s = np.exp(ln_rr_s - 1.96 * se_s)
ci_hi_s = np.exp(ln_rr_s + 1.96 * se_s)

print(f"男性 vs 女性 → 感染")
print(f"  RR = {rr_sex:.3f} (95% CI: {ci_lo_s:.3f} – {ci_hi_s:.3f})")
print(f"  OR = {or_sex:.3f}")
print(f"  p-value = {p_s:.4f}")

print(f"\n=== 分性別致死率 ===")
for sex_label in ["M", "F"]:
    infected_sex = df[(df["sex"] == sex_label) & (df["infected"] == 1)]
    deaths_sex = (infected_sex["outcome"] == "dead").sum()
    n_infected = len(infected_sex)
    cfr = deaths_sex / n_infected if n_infected > 0 else 0
    print(f"  {sex_label}: CFR = {cfr:.1%} ({deaths_sex}/{n_infected})")

print("\n解讀：")
print("- 感染風險（RR）：衡量易感性（susceptibility）— 性別是否影響感染機率")
print("- 致死率（CFR）：衡量預後（prognosis）— 感染後性別是否影響存活")
print("- 兩者是不同的問題，需要分開分析")

## 題目 4：世代研究 vs. 病例對照研究

In [ ]:
# 這是病例對照研究（case-control study）
# 研究者「先找病例，再選對照」，不是追蹤整個族群
# 分母是人為決定的（30:60），不代表真實疾病發生率
# 所以 (24/39) / (6/51) 不是真正的「風險比」

a, b, c, d = 24, 15, 6, 45
or_val = odds_ratio(a, b, c, d)

ln_or = np.log(or_val)
se_or = np.sqrt(1/a + 1/b + 1/c + 1/d)
ci_lo = np.exp(ln_or - 1.96 * se_or)
ci_hi = np.exp(ln_or + 1.96 * se_or)

oddsr_f, p_fisher = fisher_exact([[a, b], [c, d]])

print("=== 病例對照研究：沙拉 × 食物中毒 ===")
print(f"OR = {or_val:.3f} (95% CI: {ci_lo:.3f} – {ci_hi:.3f})")
print(f"Fisher 精確檢定: p = {p_fisher:.6f}")
print(f"\n解讀：吃沙拉者的感染勝算是未吃者的 {or_val:.1f} 倍")
print("CI 不包含 1 且 p 值極小 → 沙拉與食物中毒有統計顯著關聯")
print("\n若能取得全部 500 人資料 → 變成回溯性世代研究 → 可以算 RR")

## 題目 5（挑戰題）：小樣本的 Fisher vs. 卡方

In [ ]:
subset = df[(df["floor"] == 3) & (df["wing"] == "B")].copy()
print(f"3 樓 B 翼住民數：{len(subset)} 人")
print(f"感染人數：{subset['infected'].sum()} 人")

ct_sub = pd.crosstab(subset["shower_use"], subset["infected"])
print(f"\n2×2 表：")
print(ct_sub)

a_sub = int(ct_sub.loc[1, 1]) if 1 in ct_sub.index and 1 in ct_sub.columns else 0
b_sub = int(ct_sub.loc[1, 0]) if 1 in ct_sub.index and 0 in ct_sub.columns else 0
c_sub = int(ct_sub.loc[0, 1]) if 0 in ct_sub.index and 1 in ct_sub.columns else 0
d_sub = int(ct_sub.loc[0, 0]) if 0 in ct_sub.index and 0 in ct_sub.columns else 0

chi2_sub, p_chi2, dof, expected = chi2_contingency([[a_sub, b_sub], [c_sub, d_sub]])
print(f"\n期望值表：")
print(pd.DataFrame(expected.round(2),
                   index=["未使用淋浴", "使用淋浴"],
                   columns=["未感染", "感染"]))

min_exp = expected.min()
print(f"\n最小期望值 = {min_exp:.2f}", end="")
if min_exp < 5:
    print(" → < 5，卡方近似可能不準確！")
else:
    print(" → >= 5，卡方檢定適用")

oddsr_f, p_fisher = fisher_exact([[a_sub, b_sub], [c_sub, d_sub]])
print(f"\n卡方檢定: χ² = {chi2_sub:.3f}, p = {p_chi2:.4f}")
print(f"Fisher 精確檢定: p = {p_fisher:.4f}")
print(f"p-value 差距: {abs(p_chi2 - p_fisher):.4f}")
print("\n結論：")
print("- 小樣本時，Fisher 精確檢定更可靠（不依賴大樣本近似）")
print("- 卡方檢定在期望值 < 5 時可能高估或低估顯著性")
print("- 實務上，期望值 < 5 的格子超過 20% 就建議改用 Fisher")

### 解讀

**題目 4 重點：**
- **病例對照研究**不能算 RR，因為分母是研究者人為決定的，不代表真實族群的疾病發生率
- **OR** 是病例對照研究的正確效應量指標
- 若能取得全員資料，就變成世代研究，可以算 RR

**題目 5 重點：**
- 小樣本時，卡方檢定的 χ² 分布近似不夠準確
- **Fisher 精確檢定**直接計算在 H₀ 下觀察到當前或更極端結果的確切機率
- 實務法則：任何格子的期望值 < 5 時，優先使用 Fisher